<a href="https://colab.research.google.com/github/josemartinlabarca/mineria-datos-churn-telco/blob/main/lab-2-10-clustering-kmeans-jerarquico/Lab_2_10_Clustering_KMeans_Jerarquico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 Laboratorio 2.10 · Clustering: K-means y jerárquico
**IEI097 Minería de Datos · Unidad II** — José Labarca

| Ejercicio | Método | Dataset | Pregunta |
|---|---|---|---|
| 1 | K-means (partición) | Mall Customer Segmentation (Kaggle) | ¿Qué segmentos de clientes justifican comunicaciones distintas? |
| 2 | Jerárquico aglomerativo | Unsupervised Learning on Country Data (Kaggle) | ¿Con qué países se agrupa Chile y a qué nivel de la jerarquía? |

> **Regla de lectura:** un cluster significa «estas observaciones son cercanas bajo *estas* variables, *esta* transformación, *esta* distancia y *estos* parámetros». Ningún grupo de este notebook es una verdad oculta en los datos.

**Decisiones fijas en todo el notebook:** distancia euclidiana · `SEMILLA = 42` · `n_init = 10` en K-means · estandarización con `StandardScaler` salvo cuando se compara explícitamente con los datos crudos.

## 0 · Configuración

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
from scipy.spatial.distance import pdist

warnings.filterwarnings("ignore")
SEMILLA = 42
N_INIT = 10
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)

In [ ]:
# Credenciales de Kaggle desde los Secretos de Colab (icono de la llave 🔑). Nunca se escriben en el notebook.
EN_COLAB = "google.colab" in str(get_ipython())
if EN_COLAB:
    from google.colab import userdata

    def leer_secreto(*nombres):
        for n in nombres:
            try:
                return userdata.get(n)
            except Exception:
                pass
        return None

    token   = leer_secreto("KAGGLE_API_TOKEN", "KAGGLE_TOKEN")
    usuario = leer_secreto("KAGGLE_USERNAME")
    clave   = leer_secreto("KAGGLE_KEY")
    if token:
        os.environ["KAGGLE_API_TOKEN"] = token
        print("Usando KAGGLE_API_TOKEN")
    elif usuario and clave:
        os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = usuario, clave
        print("Usando KAGGLE_USERNAME + KAGGLE_KEY")
    else:
        raise ValueError("No encontré credenciales en Colab Secrets (llave 🔑). Crea KAGGLE_API_TOKEN y activa 'Acceso del notebook'.")
    import kagglehub


def descargar_csv(handle, nombre=None):
    """Descarga un dataset de Kaggle, LISTA sus archivos (no asumimos el nombre) y devuelve la ruta del CSV principal."""
    path = kagglehub.dataset_download(handle) if EN_COLAB else os.environ.get("DATA_DIR", ".")
    csvs = sorted(glob.glob(os.path.join(path, "**", "*.csv"), recursive=True))
    print("Carpeta:", path)
    print("Archivos CSV:", [os.path.relpath(c, path) for c in csvs])
    if nombre:
        csvs = [c for c in csvs if os.path.basename(c).lower() == nombre.lower()] or csvs
    print("CSV principal:", os.path.basename(csvs[0]))
    return csvs[0]

---
# Ejercicio 1 · 🛍️ K-means: segmentar clientes de una tienda
**Caso:** una tienda regional quiere comunicaciones diferenciadas. No hay «clase correcta»: hay hipótesis de segmento que hay que poder defender.

### Paso 1 · Hipótesis antes de calcular
Cargo el archivo solo para poder graficar la nube de puntos **sin colorear**; la exploración formal viene en el paso 2.

In [ ]:
ruta_mall = descargar_csv("vjchoudhary7/customer-segmentation-tutorial-in-python", "Mall_Customers.csv")
mall = pd.read_csv(ruta_mall)
mall = mall.rename(columns={"Genre": "Gender"})   # algunas copias del archivo traen 'Genre'
INC, SPE, AGE = "Annual Income (k$)", "Spending Score (1-100)", "Age"

fig, ax = plt.subplots(figsize=(7, 5))
sns.scatterplot(data=mall, x=INC, y=SPE, color="gray", s=40, ax=ax)
ax.set_title("Ingreso anual vs puntaje de gasto (sin colorear)")
plt.show()

#### 📝 Hipótesis inicial (escrita antes de ejecutar cualquier algoritmo)
Veo **5 grupos**. Evidencia visual:
- Una **nube central densa** (ingreso ≈ 40–70 k$, gasto ≈ 40–60), claramente la más poblada.
- **Cuatro «esquinas»** separadas de la nube central por zonas casi vacías: ingreso bajo/gasto alto, ingreso bajo/gasto bajo, ingreso alto/gasto alto e ingreso alto/gasto bajo.
- Las esquinas de ingreso alto se ven más dispersas que las de ingreso bajo, así que espero que sean los grupos menos compactos.

La retomo en el paso 6.

### Paso 2 · Cargar y explorar

In [ ]:
print("Dimensiones:", mall.shape)
mall.info()
print("\nNulos por columna:\n", mall.isna().sum())
print("\nCustomerID único por fila:", mall["CustomerID"].is_unique)
print("\n", mall["Gender"].value_counts())
mall.describe().round(1)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for i, col in enumerate([AGE, INC, SPE]):
    sns.histplot(mall[col], bins=20, kde=True, ax=axes[0, i]).set_title(f"Histograma · {col}")
    sns.boxplot(x=mall[col], ax=axes[1, i]).set_title(f"Box-plot · {col}")
plt.tight_layout(); plt.show()

#### 📝 Unidad de análisis y variables
- **Unidad de análisis:** el cliente (una fila = un cliente, 200 en total, sin nulos ni IDs duplicados).
- **`CustomerID` se excluye:** es un identificador correlativo; incluirlo en la distancia haría «parecidos» a clientes que solo se registraron uno después del otro.
- **`Gender` se excluye del modelo** (se conserva para describir los grupos al final). Si lo codifico 0/1 y lo meto en una distancia euclidiana, aporta un salto fijo de 1 (o de ~2 tras estandarizar) entre cualquier hombre y cualquier mujer, sin valores intermedios. Eso tiende a **partir los datos por sexo** aunque el sexo no tenga que ver con el comportamiento de compra, y además mezclaría una variable categórica en una métrica pensada para variables continuas. Además, segmentar comunicaciones por sexo no es el objetivo del caso.
- Ingreso tiene un único valor alto (137 k$) que el box-plot marca como atípico; no lo elimino porque es un cliente real y no es un error de registro.

### Paso 3 · Preparar: variables y escala
Modelo principal: **ingreso + puntaje de gasto** (2D, visualizable). Edad se suma después como variante.

In [ ]:
VARS = [INC, SPE]
X_raw = mall[VARS].copy()
print("Rangos antes de escalar:")
print(X_raw.agg(["min", "max", "mean", "std"]).T.round(1))

scaler = StandardScaler()
X_std = pd.DataFrame(scaler.fit_transform(X_raw), columns=VARS, index=X_raw.index)
print("\nDespués de StandardScaler (media ≈ 0, desviación ≈ 1):")
print(X_std.agg(["mean", "std"]).T.round(3))

#### 📝 Rangos y decisión de escala
- `Annual Income (k$)`: 15 – 137 · `Spending Score (1-100)`: 1 – 99.
- **Decisión: estandarizo.** Las dos variables están en unidades distintas (miles de dólares vs puntaje), y el rango «parecido» es una coincidencia de cómo se expresó el ingreso: si estuviera en dólares o en pesos, dominaría la distancia por completo. Estandarizar hace que cada variable pese lo mismo sin depender de la unidad elegida. Guardo `X_raw` y `X_std` para el mini-experimento del paso 5.

### Paso 4 · Ajustar K-means y explorar k

In [ ]:
ks = range(2, 11)
inercias, siluetas = [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=SEMILLA).fit(X_std)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_std, km.labels_))

res_k = pd.DataFrame({"k": list(ks), "inercia": inercias, "silueta": siluetas}).round(3)
display(res_k)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(res_k.k, res_k.inercia, "o-"); axes[0].set(title="Método del codo (inercia)", xlabel="k", ylabel="inercia")
axes[1].plot(res_k.k, res_k.silueta, "o-", color="tab:green"); axes[1].set(title="Coeficiente de silueta", xlabel="k", ylabel="silueta")
for ax in axes: ax.axvline(5, ls="--", c="gray")
plt.tight_layout(); plt.show()

#### 📝 ¿Por qué la inercia siempre baja al subir k?
La inercia es la suma de distancias cuadráticas de cada punto a su centroide. Con un centroide más, cada punto puede quedarse con el centroide que ya tenía o pasarse a uno más cercano: nunca empeora. En el extremo, con k = n (un centroide por cliente) la inercia es 0. Por eso **la curva sola no puede elegir k**: siempre «premia» más grupos. Solo sirve para buscar el punto donde agregar un grupo deja de reducir mucho la inercia (el codo), y ese punto es una lectura visual subjetiva. La silueta, en cambio, compara la cohesión interna con la separación respecto del grupo vecino, y sí puede empeorar al agregar grupos.

### Paso 5 · Evaluar con más de una evidencia

**(a) Estructura interna:** la tabla del paso 4 se lee junto con la caída relativa de inercia por cada grupo añadido.

In [ ]:
res_k["caida_inercia_%"] = (-res_k.inercia.pct_change() * 100).round(1)
display(res_k)
print("k con mayor silueta:", int(res_k.loc[res_k.silueta.idxmax(), "k"]))

**(b) Estabilidad:** (1) repito el ajuste con otras semillas; (2) reajusto sobre submuestras del 80 % y comparo, sobre esos mismos clientes, con la partición completa. Uso el índice de Rand ajustado (ARI: 1 = misma partición, ≈0 = acuerdo por azar), que no depende del número que reciba cada grupo.

In [ ]:
K = 5
base = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEMILLA).fit(X_std)
lab_base = pd.Series(base.labels_, index=X_std.index)

ari_semillas = {s: adjusted_rand_score(lab_base, KMeans(n_clusters=K, n_init=N_INIT, random_state=s).fit_predict(X_std))
                for s in [0, 1, 7, 123, 2024]}
print("ARI vs semilla 42 →", {s: round(v, 3) for s, v in ari_semillas.items()})

ari_sub = []
for s in range(10):
    m = X_std.sample(frac=0.8, random_state=s)
    lab_m = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEMILLA).fit_predict(m)
    ari_sub.append(adjusted_rand_score(lab_base.loc[m.index], lab_m))
print(f"ARI en 10 submuestras del 80 %: media {np.mean(ari_sub):.3f} · mín {np.min(ari_sub):.3f}")

# misma prueba para k=4 y k=6, para ver si la estabilidad es propia de k=5
for k_alt in [4, 6]:
    b = KMeans(n_clusters=k_alt, n_init=N_INIT, random_state=SEMILLA).fit_predict(X_std)
    a = [adjusted_rand_score(pd.Series(b, index=X_std.index).loc[m.index],
                             KMeans(n_clusters=k_alt, n_init=N_INIT, random_state=SEMILLA).fit_predict(m))
         for m in (X_std.sample(frac=0.8, random_state=s) for s in range(10))]
    print(f"  k={k_alt}: ARI medio en submuestras {np.mean(a):.3f} · mín {np.min(a):.3f}")

**Mini-experimento de escala:** mismo k = 5, con y sin estandarizar. Lo repito en tres representaciones para ver *cuándo* importa la escala.

In [ ]:
def ari_escala(df_vars):
    crudo = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEMILLA).fit_predict(df_vars)
    estd  = KMeans(n_clusters=K, n_init=N_INIT, random_state=SEMILLA).fit_predict(StandardScaler().fit_transform(df_vars))
    return crudo, estd, adjusted_rand_score(crudo, estd)

# A) ingreso (k$) + gasto
crudoA, stdA, ariA = ari_escala(mall[[INC, SPE]])
# B) edad + ingreso + gasto
crudoB, stdB, ariB = ari_escala(mall[[AGE, INC, SPE]])
# C) mismo caso A pero con el ingreso expresado en dólares (×1000): solo cambia la unidad
mall_usd = mall[[INC, SPE]].assign(**{INC: mall[INC] * 1000})
crudoC, stdC, ariC = ari_escala(mall_usd)

print(f"A) ingreso k$ + gasto          → ARI crudo vs estandarizado = {ariA:.3f}")
print(f"B) edad + ingreso + gasto      → ARI crudo vs estandarizado = {ariB:.3f}")
print(f"C) ingreso en USD + gasto      → ARI crudo vs estandarizado = {ariC:.3f}")
print("\nTamaños de grupo en C:")
print(pd.DataFrame({"crudo_USD": pd.Series(crudoC).value_counts().sort_index().values,
                    "estandarizado": pd.Series(stdC).value_counts().sort_index().values}))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(mall[INC], mall[SPE], c=crudoC, cmap="tab10", s=30); axes[0].set_title("C) sin estandarizar (ingreso en USD)")
axes[1].scatter(mall[INC], mall[SPE], c=stdC, cmap="tab10", s=30); axes[1].set_title("C) estandarizado")
for ax in axes: ax.set(xlabel=INC, ylabel=SPE)
plt.tight_layout(); plt.show()

In [ ]:
# Variante: sumar la edad como tercera variable (estandarizada) y ver si mejora la estructura
X3 = StandardScaler().fit_transform(mall[[AGE, INC, SPE]])
sil3 = {k: round(silhouette_score(X3, KMeans(n_clusters=k, n_init=N_INIT, random_state=SEMILLA).fit_predict(X3)), 3) for k in range(2, 9)}
print("Silueta con edad + ingreso + gasto:", sil3)
print("Silueta máxima con 2 variables:", res_k.silueta.max())

#### 📝 Elección de k y evidencias
**Elijo k = 5.**
1. **Estructura interna:** k = 5 tiene la **silueta máxima** del rango 2–10 (≈ 0,55) y es el último k con una caída de inercia grande (≈ 40 % de 4 → 5; de 5 → 6 cae solo ≈ 16 %). Codo y silueta apuntan al mismo lugar.
2. **Estabilidad:** las otras cinco semillas reproducen exactamente la misma partición (ARI = 1) y en submuestras del 80 % el ARI medio es muy alto; k = 4 y k = 6 son menos estables en esas mismas submuestras (ver salida), lo que sugiere que con ellos K-means parte o fusiona grupos de manera más arbitraria.
3. **Utilidad:** los 5 grupos se pueden describir con una frase cada uno y cada uno pide una acción distinta (paso 6). Con k = 4 se fusionan dos esquinas con comportamientos opuestos; con k = 6 se parte la nube central en dos mitades sin una diferencia de negocio clara.

**Mini-experimento de escala:**
- **A (ingreso k$ + gasto):** la partición es **idéntica** con y sin estandarizar. No es que la escala no importe, es que en este caso concreto ambas variables ya tienen desviaciones casi iguales (≈ 26 y ≈ 26), así que estandarizar casi no cambia sus pesos relativos.
- **B (con edad):** sí cambia; la edad tiene otra dispersión y el peso relativo de cada variable se reordena.
- **C (ingreso en USD):** cambiar solo la *unidad* del ingreso destruye la segmentación cruda: la distancia queda dominada por el ingreso y K-means corta en franjas verticales de ingreso, ignorando el gasto. Estandarizado, el resultado es el mismo que en A.

Conclusión: el resultado sin estandarizar depende de una elección arbitraria (la unidad de medida); el estandarizado no. Por eso uso la versión estandarizada.

### Paso 6 · Interpretar y decidir

In [ ]:
mall["cluster"] = lab_base.values
perfil = (mall.groupby("cluster")
              .agg(n=("CustomerID", "count"),
                   edad_media=(AGE, "mean"), edad_mediana=(AGE, "median"),
                   ingreso_medio=(INC, "mean"), ingreso_mediana=(INC, "median"),
                   gasto_medio=(SPE, "mean"), gasto_mediana=(SPE, "median"),
                   pct_mujeres=("Gender", lambda g: (g == "Female").mean() * 100))
              .round(1))

# Nombre automático según la posición del grupo (así no depende del número que K-means le asigne)
def nivel(v, bajo, alto):
    return "bajo" if v < bajo else ("alto" if v > alto else "medio")

NOMBRES = {
    ("alto", "alto"):   "Premium activos",
    ("alto", "bajo"):   "Acomodados cautelosos",
    ("bajo", "alto"):   "Jóvenes entusiastas",
    ("bajo", "bajo"):   "Ahorradores ajustados",
    ("medio", "medio"): "Clientes promedio",
}
ACCIONES = {
    "Premium activos":       "Programa de fidelización VIP: acceso anticipado a lanzamientos y beneficios exclusivos (cuidar retención, no descuentos).",
    "Acomodados cautelosos": "Comunicación de calidad y valor (garantías, productos premium, asesoría personalizada) para activar su capacidad de gasto.",
    "Jóvenes entusiastas":   "Promociones en redes sociales, novedades y medios de pago en cuotas; vigilar que la oferta no empuje a sobreendeudamiento.",
    "Ahorradores ajustados": "Ofertas de precio bajo y productos básicos; mensajes de ahorro, baja frecuencia de contacto.",
    "Clientes promedio":     "Newsletter general con ofertas por temporada; base para probar A/B de mensajes.",
}
perfil["segmento"] = [NOMBRES.get((nivel(r.ingreso_medio, 40, 70), nivel(r.gasto_medio, 35, 65)), "Sin nombre")
                      for r in perfil.itertuples()]
perfil["accion"] = perfil["segmento"].map(ACCIONES)
display(perfil.drop(columns="accion"))

In [ ]:
# Centroides: cluster_centers_ está en el espacio estandarizado → se vuelven a unidades originales con inverse_transform
centros = pd.DataFrame(scaler.inverse_transform(base.cluster_centers_), columns=VARS).round(1)
centros["segmento"] = perfil["segmento"].values
display(centros)

fig, ax = plt.subplots(figsize=(9, 6))
for c, r in perfil.iterrows():
    sub = mall[mall.cluster == c]
    ax.scatter(sub[INC], sub[SPE], s=40, label=f"{r.segmento} (n={r.n})")
ax.scatter(centros[INC], centros[SPE], marker="X", s=250, c="black", label="centroides")
ax.set(xlabel=INC, ylabel=SPE, title="Segmentos K-means (k=5, datos estandarizados, ejes en unidades originales)")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()

In [ ]:
with pd.option_context("display.max_colwidth", None):
    display(perfil[["segmento", "n", "accion"]])

#### 📝 Segmentos, acciones y reflexión
La tabla de perfiles (en unidades originales) queda arriba; cada segmento se nombra por su posición en ingreso y gasto y tiene una acción de comunicación distinta para **probar** (no para aplicar a ciegas: lo correcto es medir respuesta, por ejemplo con un A/B test por segmento).

**Reflexión:**
- **La hipótesis coincidió:** los 5 grupos que vi a ojo (nube central + 4 esquinas) son los que K-means recupera, con la silueta máxima en k = 5.
- **Lo que me sorprendió:** que estandarizar no cambiara nada en 2D, hasta que entendí que era porque ambas variables ya tenían casi la misma desviación. El experimento con el ingreso en dólares mostró que era suerte de la unidad, no robustez del método.
- También que la **edad**, que no entró al modelo, separa bien a los grupos: los de gasto alto son claramente más jóvenes. Al sumar edad como tercera variable, la silueta máxima baja de ≈ 0,55 a ≈ 0,43 y los grupos se vuelven menos nítidos, así que preferí el modelo 2D, más interpretable.
- **Cuidado:** el *Spending Score* lo asigna el centro comercial con una regla que no conocemos; los segmentos describen ese puntaje, no la «personalidad» del cliente. Tampoco uso el sexo para diseñar las acciones.

---
# Ejercicio 2 · 🌍 Dendrograma: ¿con qué países se parece Chile?
**Caso:** una organización de cooperación quiere saber qué países se parecen en indicadores socioeconómicos y de salud, y a qué nivel de detalle conviene agruparlos.

### Paso 1 · Hipótesis y muestra de trabajo

In [ ]:
ruta_pais = descargar_csv("rohan0301/unsupervised-learning-on-country-data", "Country-data.csv")
paises = pd.read_csv(ruta_pais).set_index("country")   # el nombre del país pasa a ser índice (etiqueta), no variable
print(paises.shape)
print(sorted(paises.index.unique()))

In [ ]:
AMERICAS = ["Antigua and Barbuda", "Argentina", "Bahamas", "Barbados", "Belize", "Bolivia", "Brazil", "Canada",
            "Chile", "Colombia", "Costa Rica", "Dominican Republic", "Ecuador", "El Salvador", "Grenada",
            "Guatemala", "Guyana", "Haiti", "Jamaica", "Mexico", "Panama", "Paraguay", "Peru",
            "St. Vincent and the Grenadines", "Suriname", "United States", "Uruguay", "Venezuela"]
REFERENCIA_OCDE = ["Spain", "Portugal", "Greece", "Poland", "New Zealand", "South Korea"]

faltantes = [p for p in AMERICAS + REFERENCIA_OCDE if p not in paises.index]
print("No están en el dataset:", faltantes)
seleccion = [p for p in AMERICAS + REFERENCIA_OCDE if p in paises.index]
sub = paises.loc[seleccion]
print(f"Subconjunto: {len(sub)} países (incluye Chile: {'Chile' in sub.index})")

#### 📝 Criterio de selección e hipótesis
**Criterio:** todos los países de **América** presentes en el dataset (27; México no viene en el archivo) **más 6 países de referencia fuera de la región** con ingresos medios-altos o altos (España, Portugal, Grecia, Polonia, Nueva Zelanda y Corea del Sur). Total: 33 países. La región permite ver con quién se parece Chile «en su barrio»; las referencias permiten probar si, en estos indicadores, Chile se parece más a sus vecinos o a economías con un ingreso parecido de otros continentes.

**Hipótesis:** Chile se agrupará con **Uruguay, Argentina y Costa Rica** (mortalidad infantil baja, esperanza de vida alta, ingreso medio-alto dentro de la región) y, a un nivel más alto de la jerarquía, con **Polonia y Portugal**. Espero que EE. UU. y Canadá queden en un grupo propio con los países de referencia más ricos, y que **Haití** quede aislado por su mortalidad infantil.

### Paso 2 · Cargar y explorar

In [ ]:
print("Nulos en el dataset completo:", int(paises.isna().sum().sum()))
display(sub.describe().T.round(1))
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
for ax, col in zip(axes.ravel(), sub.columns):
    sns.boxplot(y=sub[col], ax=ax); ax.set_title(col)
axes.ravel()[-1].axis("off")
plt.suptitle("Distribución de cada variable en el subconjunto (unidades originales)"); plt.tight_layout(); plt.show()

plt.figure(figsize=(8, 6))
sns.heatmap(sub.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlación entre variables (subconjunto)"); plt.show()

#### 📝 Magnitudes
- Unidades: `child_mort` en muertes por cada 1000 nacidos; `exports`, `imports` y `health` en **% del PIB**; `income` y `gdpp` en **dólares** por persona; `inflation` en % anual; `life_expec` en años; `total_fer` en hijos por mujer.
- **`income` y `gdpp` dominarían una distancia euclidiana sin preparar:** su desviación es de ≈ 11 000–12 500, frente a ≈ 0,6 de la fecundidad o ≈ 8 de la esperanza de vida. En crudo, la distancia entre dos países sería prácticamente la diferencia de ingreso y PIB per cápita.
- El mapa de calor muestra bloques muy correlacionados (`income`–`gdpp`, y `child_mort`–`total_fer`–`life_expec` con signo opuesto). Eso significa que «riqueza» y «salud» entran casi dos veces cada una en la distancia; lo dejo como limitación declarada.
- Haití es un atípico extremo en mortalidad infantil y esperanza de vida, y Venezuela en inflación.

### Paso 3 · Preparar: ¿cuánto pesa cada variable?

In [ ]:
etiquetas = sub.index.tolist()                         # nombres separados de la matriz numérica
M_raw = sub.values
M_std = StandardScaler().fit_transform(sub)
print("Desviación por variable · crudo:", dict(zip(sub.columns, sub.std().round(1))))
print("Desviación por variable · estandarizado:", dict(zip(sub.columns, M_std.std(axis=0).round(2))))

#### 📝 Predicción
Sin estandarizar, el dendrograma será casi un **orden por ingreso/PIB per cápita**: los países quedarán agrupados por franjas de riqueza y las diferencias en salud, inflación o comercio no se notarán. Al estandarizar espero que aparezcan grupos que mezclen salud y economía, y que los atípicos (Haití por salud, Venezuela por inflación) se separen del resto.

### Paso 4 · Construir la jerarquía y leer el dendrograma

In [ ]:
ENLACES = ["single", "complete", "average", "ward"]
Z = {m: linkage(M_std, method=m, metric="euclidean") for m in ENLACES}

fig, axes = plt.subplots(1, 4, figsize=(22, 10))
for ax, m in zip(axes, ENLACES):
    dendrogram(Z[m], labels=etiquetas, orientation="left", leaf_font_size=9, ax=ax)
    ax.set_title(f"Enlace {m} · estandarizado", fontsize=13)
    ax.set_xlabel("altura de fusión (disimilitud)")
    for t in ax.get_ymajorticklabels():
        if t.get_text() == "Chile": t.set_color("red"); t.set_fontweight("bold")
plt.tight_layout(); plt.show()

In [ ]:
# Ajuste de fidelidad: correlación cofenética (qué tan bien conserva cada dendrograma las distancias originales)
d_std = pdist(M_std)
display(pd.DataFrame({"enlace": ENLACES,
                      "corr_cofenetica": [round(cophenet(Z[m], d_std)[0], 3) for m in ENLACES]}))

In [ ]:
# Mismo enlace preferido (ward) con los datos SIN estandarizar
Z_raw = linkage(M_raw, method="ward")
fig, axes = plt.subplots(1, 2, figsize=(18, 10))
for ax, (zz, tit) in zip(axes, [(Z["ward"], "ward · estandarizado"), (Z_raw, "ward · SIN estandarizar")]):
    dendrogram(zz, labels=etiquetas, orientation="left", leaf_font_size=9, ax=ax); ax.set_title(tit, fontsize=13)
    for t in ax.get_ymajorticklabels():
        if t.get_text() == "Chile": t.set_color("red"); t.set_fontweight("bold")
plt.tight_layout(); plt.show()

# ¿El crudo agrupa por riqueza? Corto ambos árboles en 4 grupos y miro el rango de gdpp de cada grupo
for nombre, zz in [("SIN estandarizar", Z_raw), ("estandarizado", Z["ward"])]:
    g = fcluster(zz, t=4, criterion="maxclust")
    t = sub.assign(g=g).groupby("g").agg(n=("gdpp", "size"), gdpp_min=("gdpp", "min"), gdpp_max=("gdpp", "max"),
                                         mort_inf_min=("child_mort", "min"), mort_inf_max=("child_mort", "max"))
    print(f"\nWard {nombre}, 4 grupos (ordenados por gdpp):"); display(t.sort_values("gdpp_min"))
g_raw = fcluster(Z_raw, t=4, criterion="maxclust")
print("Sin estandarizar, Haití queda con:", ", ".join(p for p in sub.index[g_raw == g_raw[etiquetas.index("Haiti")]] if p != "Haiti"))

#### 📝 Diferencias de forma
- **Single:** forma de «escalera» (efecto cadena). Va pegando los países de a uno a un bloque grande, y al final cuelgan solos Haití, Venezuela y EE. UU. Sirve para detectar atípicos, pero no entrega grupos útiles.
- **Complete:** grupos más compactos y alturas más parejas, porque fusiona según el par **más lejano**. Aun así, su primer corte separa solo a Haití, y el siguiente a los países ricos.
- **Average:** intermedio (aísla a Haití, EE. UU. y Venezuela en los primeros cortes); es el que mejor conserva las distancias originales (correlación cofenética más alta), pero también aísla los atípicos y deja un bloque latinoamericano enorme.
- **Ward:** el más equilibrado; produce ramas de tamaño parecido y saltos claros arriba del árbol, porque cada fusión minimiza el aumento de varianza dentro de los grupos. Su cofenética es la más baja: «deforma» más las distancias a cambio de grupos compactos.
- **Sin estandarizar:** se cumplió la predicción. Al cortar el árbol crudo en 4 grupos, cada grupo es una **franja de PIB per cápita que no se traslapa con las demás** (ver tabla), mientras que en el estandarizado los rangos sí se mezclan. La salud casi no influye: Haití, con una mortalidad infantil diez veces mayor que la del resto, deja de estar aislado y queda junto a Costa Rica o Colombia solo porque su ingreso cae en la misma franja baja.

### Paso 5 · Cortar la jerarquía
Elijo **ward** (grupos equilibrados, lo que necesita la organización para trabajar por grupos) y pruebo cortes de 2 a 5 grupos.

In [ ]:
Zw = Z["ward"]
alturas = Zw[:, 2]
cortes = []
for k in [2, 3, 4, 5]:
    lab = fcluster(Zw, t=k, criterion="maxclust")
    cortes.append({"k": k, "silueta": round(silhouette_score(M_std, lab), 3),
                   "altura_fusion_que_se_evita": round(alturas[-(k - 1)], 2),
                   "salto_hasta_esa_fusion": round(alturas[-(k - 1)] - alturas[-k], 2),
                   "tamaños": sorted(pd.Series(lab).value_counts().tolist(), reverse=True)})
display(pd.DataFrame(cortes))

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, 11), alturas[::-1][:10], "o-")
plt.xlabel("número de grupos antes de la fusión"); plt.ylabel("altura"); plt.title("Altura de las últimas fusiones (ward)")
plt.show()

In [ ]:
K_CORTE = 4
h_corte = (alturas[-(K_CORTE - 1)] + alturas[-K_CORTE]) / 2      # entre la fusión que se evita y la anterior
fig, ax = plt.subplots(figsize=(10, 10))
dendrogram(Zw, labels=etiquetas, orientation="left", leaf_font_size=10, color_threshold=h_corte, ax=ax)
ax.axvline(h_corte, ls="--", c="red", lw=2, label=f"corte k={K_CORTE} (altura ≈ {h_corte:.2f})")
# referencia: dónde estaría el corte de 3 grupos
h3 = (alturas[-2] + alturas[-3]) / 2
ax.axvline(h3, ls=":", c="gray", lw=1.5, label=f"corte k=3 (altura ≈ {h3:.2f})")
for t in ax.get_ymajorticklabels():
    if t.get_text() == "Chile": t.set_color("red"); t.set_fontweight("bold")
ax.set_title("Ward · estandarizado · corte elegido"); ax.set_xlabel("altura de fusión (disimilitud)"); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

> Con `orientation="left"` el eje de alturas es horizontal, por eso la línea de corte se dibuja con `axvline` (en un dendrograma vertical sería `axhline`).

#### 📝 Corte elegido: k = 4 (ward)
- **Estructura:** k = 3 tiene la silueta más alta y el salto de altura más grande; sobre esa evidencia sola, 3 sería la respuesta. Pero con k = 3 Chile queda en un bloque de 26 de 33 países que mezcla economías muy distintas (Chile con Guatemala o Barbados), así que el corte no responde la pregunta.
- **k = 4** es el siguiente salto claro del árbol y parte ese bloque en dos grupos con una diferencia interpretable (apertura comercial e ingreso; ver perfiles abajo). Su silueta es más baja que la de k = 3 pero parecida a la de k = 5, y k = 5 solo desprende un país más sin aportar una idea nueva.
- **Utilidad:** 4 grupos se pueden describir uno por uno y usar para planificar.
- Recuerdo que **la altura mide disimilitud, no calidad:** que una fusión esté «alta» indica que se juntan grupos distintos, no que el grupo sea mejor o peor.

### Paso 6 · Interpretar y comparar con K-means

In [ ]:
sub_c = sub.copy()
sub_c["jerarquico"] = fcluster(Zw, t=K_CORTE, criterion="maxclust")
for g, miembros in sub_c.groupby("jerarquico").groups.items():
    print(f"Grupo {g} ({len(miembros)}): {', '.join(miembros)}")

perfil_p = sub_c.groupby("jerarquico").mean().round(1)
perfil_p.insert(0, "n", sub_c["jerarquico"].value_counts().sort_index())
display(perfil_p)

g_chile = sub_c.loc["Chile", "jerarquico"]
print(f"\nChile está en el grupo {g_chile} con:", ", ".join(p for p in sub_c.index[sub_c.jerarquico == g_chile] if p != "Chile"))
dist_chile = pd.Series(np.linalg.norm(M_std - M_std[etiquetas.index("Chile")], axis=1), index=etiquetas).drop("Chile")
print("Vecinos más cercanos de Chile (distancia euclidiana estandarizada):")
print(dist_chile.nsmallest(5).round(2))

In [ ]:
sub_c["kmeans"] = KMeans(n_clusters=K_CORTE, n_init=N_INIT, random_state=SEMILLA).fit_predict(M_std)
display(pd.crosstab(sub_c["jerarquico"], sub_c["kmeans"], rownames=["ward"], colnames=["K-means"]))
print("ARI ward vs K-means:", round(adjusted_rand_score(sub_c.jerarquico, sub_c.kmeans), 3))
for g, miembros in sub_c.groupby("kmeans").groups.items():
    print(f"K-means {g} ({len(miembros)}): {', '.join(miembros)}")
g_km = sub_c.loc["Chile", "kmeans"]
print("\nCon K-means, Chile queda con:", ", ".join(p for p in sub_c.index[sub_c.kmeans == g_km] if p != "Chile"))

#### 📝 Conclusión (respuesta a la pregunta guía)
1. **¿Con quién se agrupa Chile?** Con el corte de 4 grupos (ward), Chile queda en el grupo de **economías latinoamericanas de ingreso medio con baja apertura comercial relativa**: Argentina, Brasil, Colombia, Costa Rica, Perú, Uruguay, Ecuador, República Dominicana, entre otros. Sus vecinos más cercanos son **Uruguay y Costa Rica**, lo que confirma parte de mi hipótesis. Polonia y Corea **no** quedan con Chile a este nivel, sino en el grupo de economías más abiertas al comercio; solo se juntan con Chile más arriba (corte de 3 grupos). Los países OCDE más ricos, EE. UU. y Canadá forman su propio grupo, y Haití queda aislado, como predije.
2. **Qué cambia entre métodos:** ambos coinciden en el grupo de países ricos (al que K-means suma las Bahamas, Polonia y Corea) y en aislar Haití. La diferencia está en cómo parten el bloque latinoamericano: K-means arma un núcleo más chico alrededor de Chile (Argentina, Brasil, Colombia, Costa Rica, Perú, Uruguay…) y manda a Centroamérica y el Caribe al otro grupo. El ARI es moderado. Es lógico: ward fusiona minimizando el aumento de varianza *paso a paso* (la decisión de cada fusión ya no se revisa), mientras K-means minimiza la misma inercia pero optimizando *toda la partición a la vez*.
3. **Qué presentaría:** el **dendrograma de ward**, porque muestra los niveles: Chile está con sus vecinos sudamericanos a un nivel fino y con Polonia y Corea a un nivel más grueso. Eso responde «a qué nivel» mejor que una sola partición plana.
4. **Limitaciones:** los grupos dependen de la muestra elegida (33 países, no los 167) y de que las variables correlacionadas cuentan doble (riqueza y salud); los datos son de un solo año. **No** se puede concluir que los países de un grupo sean «mejores» o «peores», ni que compartan causas o políticas: solo que se parecen en estos 9 indicadores, estandarizados, con distancia euclidiana y enlace ward.

---
## 🚀 Cierre del ciclo de razonamiento
- **¿Qué idea de «grupo» usa cada método?** K-means: puntos cercanos a un mismo centroide (grupos convexos, de tamaño parecido, k fijado de antemano). Jerárquico: grupos que se van fusionando según un criterio de enlace; *single* une por el par más cercano (cadenas), *complete* por el más lejano (compactos), *average* por el promedio y *ward* por el menor aumento de varianza (el más parecido a K-means).
- **¿Qué decisiones de representación cambiaron los resultados?** La escala fue la decisión que más pesó: en ambos ejercicios, sin estandarizar, la variable expresada en la unidad más grande (dólares) dictó los grupos. También la selección de variables (incluir la edad bajó la silueta) y, en países, la muestra elegida y el tipo de enlace.
- **¿Qué grupos usaría para decidir?** Los 5 segmentos de clientes, porque son estables entre semillas y submuestras y cada uno pide una acción distinta (para validarlas con pruebas reales). Los grupos de países los usaría solo para explorar y elegir comparaciones, no para decidir por sí solos.